In [1]:
import numpy as np
import random
import joblib
import pandas as pd
import time
import math

random.seed(None)
np.random.seed(None)

# -------------------------
# Load models (unchanged)
# -------------------------
VP0_xgboost_model = joblib.load("VP0Org1_xgboost_model.pkl")
VP1_xgboost_model = joblib.load("VP1Org1_xgboost_model.pkl")
VP2_xgboost_model = joblib.load("VP0Org2_xgboost_model.pkl")
VP3_xgboost_model = joblib.load("VP1Org2_xgboost_model.pkl")

CP0_xgboost_model = joblib.load("CP0Org1_xgboost_model.pkl")
CP1_xgboost_model = joblib.load("CP1Org1_xgboost_model.pkl")
CP2_xgboost_model = joblib.load("CP0Org2_xgboost_model.pkl")
CP3_xgboost_model = joblib.load("CP1Org2_xgboost_model.pkl")

LA_tree_model = joblib.load("latancy_forest_model.pkl")

# -------------------------
# Parameters
# -------------------------
n = 1000  # Number of transactions
r = 128 # send rate
m = 4     # peers

lb = 2
ub = 20
cb = 9 * 1024 * 1024  # 99MB max block bytes
BW = [250,250,250, 250]

# s is in KB as feature (all tx same size)
s = 2

# Derived Parameters (as requested)
nb = math.ceil(n / lb) + 1

# -------------------------
# Helpers
# -------------------------
def block_bytes_from_tx_count(tx_count: int) -> int:
    return int(tx_count * s * 1024)

def tx_column_counts(chromosome):
    return np.sum(np.array(chromosome, dtype=np.int8), axis=0)

def repair_assignment(chromosome):
    chrom = [row[:] for row in chromosome] 

    col_sum = tx_column_counts(chrom)
    missing = np.where(col_sum == 0)[0].tolist()
    duplicates = np.where(col_sum > 1)[0].tolist()

    for tx in duplicates:
        ones_blocks = [j for j in range(nb) if chrom[j][tx] == 1]
        for j in ones_blocks[1:]:
            chrom[j][tx] = 0

    col_sum = tx_column_counts(chrom)
    missing = np.where(col_sum == 0)[0].tolist()

    block_sizes = [sum(row) for row in chrom]

    def can_add_to_block(j):
        if block_sizes[j] >= ub:
            return False
        new_bytes = block_bytes_from_tx_count(block_sizes[j] + 1)
        return new_bytes <= cb

    for tx in missing:
        placed = False
        for j in range(nb):
            if can_add_to_block(j):
                chrom[j][tx] = 1
                block_sizes[j] += 1
                placed = True
                break
        if not placed:
            return None

    def move_one_tx_out(from_block):
        txs = [i for i in range(n) if chrom[from_block][i] == 1]
        if not txs:
            return True
        tx = random.choice(txs)
        chrom[from_block][tx] = 0
        block_sizes[from_block] -= 1

        for to_block in range(nb):
            if to_block == from_block:
                continue
            if can_add_to_block(to_block):
                chrom[to_block][tx] = 1
                block_sizes[to_block] += 1
                return True

        chrom[from_block][tx] = 1
        block_sizes[from_block] += 1
        return False

    for j in range(nb):
        while block_sizes[j] > 0 and (block_sizes[j] > ub or block_bytes_from_tx_count(block_sizes[j]) > cb):
            ok = move_one_tx_out(j)
            if not ok:
                return None

    small_blocks = [j for j in range(nb) if 0 < block_sizes[j] < lb]
    big_blocks = [j for j in range(nb) if block_sizes[j] > lb]

    for sb in small_blocks:
        while 0 < block_sizes[sb] < lb:
            if not big_blocks:
                merged = False
                txs = [i for i in range(n) if chrom[sb][i] == 1]
                for bb in range(nb):
                    if bb == sb:
                        continue
                    if block_sizes[bb] + len(txs) <= ub and block_bytes_from_tx_count(block_sizes[bb] + len(txs)) <= cb:
                        for tx in txs:
                            chrom[sb][tx] = 0
                            chrom[bb][tx] = 1
                        block_sizes[bb] += len(txs)
                        block_sizes[sb] = 0
                        merged = True
                        break
                if not merged:
                    return None
                break

            donor = random.choice(big_blocks)
            donor_txs = [i for i in range(n) if chrom[donor][i] == 1]
            if not donor_txs:
                big_blocks.remove(donor)
                continue

            tx = random.choice(donor_txs)
            chrom[donor][tx] = 0
            chrom[sb][tx] = 1
            block_sizes[donor] -= 1
            block_sizes[sb] += 1

            if block_sizes[donor] <= lb and donor in big_blocks:
                big_blocks.remove(donor)

    col_sum = tx_column_counts(chrom)
    if not np.all(col_sum == 1):
        return None

    for j in range(nb):
        if block_sizes[j] == 0:
            continue
        if block_sizes[j] < lb or block_sizes[j] > ub:
            return None
        if block_bytes_from_tx_count(block_sizes[j]) > cb:
            return None

    return chrom

def tx_to_block_map(chrom):
    """
    Returns a list tx_block[tx] = block_index, or None if invalid (missing/duplicate).
    Assumes chrom is already repaired; still validates.
    """
    tx_block = [-1] * n
    for b in range(nb):
        row = chrom[b]
        for tx in range(n):
            if row[tx] == 1:
                if tx_block[tx] != -1:
                    return None  # duplicate
                tx_block[tx] = b
    if any(x == -1 for x in tx_block):
        return None  # missing
    return tx_block

# -------------------------
# GA chromosome creation
# -------------------------
def create_chromosome():
    chromosome = [[0] * n for _ in range(nb)]
    for i in range(n):
        block = random.randint(0, nb - 1)
        chromosome[block][i] = 1
    chromosome = repair_assignment(chromosome)
    return chromosome

# -------------------------
# Fitness (N-model objective)
# -------------------------
def calculate_fitness(chromosome):
    """
    N-Model objective: min sum_j t_j
    where t_j = max_k(VT_jk + CT_jk) + LA_j
    Infeasible -> inf
    """
    chrom = repair_assignment([row[:] for row in chromosome])
    if chrom is None:
        return float("inf")

    block_sizes = [sum(block) for block in chrom]
    total_time = 0.0

    for j in range(nb):
        bsz = block_sizes[j]
        if bsz == 0:
            continue

        if bsz < lb or bsz > ub:
            return float("inf")
        if block_bytes_from_tx_count(bsz) > cb:
            return float("inf")

        # Latency df (NO Bandwidth, per assumption)
        df_la = pd.DataFrame({
            "Block_size": [bsz],
            "Size_of_trx_in_block": [s],
            "Send_Rate": [r],
        })
        latency = LA_tree_model.predict(df_la)[0]

        # VT/CT per-peer df (uses BW[k])
        df0 = pd.DataFrame({"Block_size":[bsz],"Size_of_trx_in_block":[s],"Send_Rate":[r],"Bandwidth":[BW[0]]})
        df1 = pd.DataFrame({"Block_size":[bsz],"Size_of_trx_in_block":[s],"Send_Rate":[r],"Bandwidth":[BW[1]]})
        df2 = pd.DataFrame({"Block_size":[bsz],"Size_of_trx_in_block":[s],"Send_Rate":[r],"Bandwidth":[BW[2]]})
        df3 = pd.DataFrame({"Block_size":[bsz],"Size_of_trx_in_block":[s],"Send_Rate":[r],"Bandwidth":[BW[3]]})

        vt0 = VP0_xgboost_model.predict(df0)[0]
        vt1 = VP1_xgboost_model.predict(df1)[0]
        vt2 = VP2_xgboost_model.predict(df2)[0]
        vt3 = VP3_xgboost_model.predict(df3)[0]

        ct0 = CP0_xgboost_model.predict(df0)[0]
        ct1 = CP1_xgboost_model.predict(df1)[0]
        ct2 = CP2_xgboost_model.predict(df2)[0]
        ct3 = CP3_xgboost_model.predict(df3)[0]

        max_storing_time = max(vt0 + ct0, vt1 + ct1, vt2 + ct2, vt3 + ct3)
        total_time += (max_storing_time + latency)

    return float(total_time)

# -------------------------
# MERGE mutation
# -------------------------
def mutate(chromosome):
    """
    MERGE mutation:
    - Pick two non-empty blocks A and B
    - If size(A)+size(B) <= ub and bytes constraint holds, move ALL tx from B -> A
    - B becomes empty (allowed)
    - Then repair (to keep feasibility)
    """
    chrom = [row[:] for row in chromosome]

    block_sizes = [sum(row) for row in chrom]
    non_empty = [i for i, sz in enumerate(block_sizes) if sz > 0]

    if len(non_empty) < 2:
        fixed = repair_assignment(chrom)
        return fixed if fixed is not None else chromosome

    for _ in range(30):
        b1, b2 = random.sample(non_empty, 2)

        s1 = block_sizes[b1]
        s2 = block_sizes[b2]
        new_size = s1 + s2

        if new_size > ub:
            continue
        if block_bytes_from_tx_count(new_size) > cb:
            continue

        for tx in range(n):
            if chrom[b2][tx] == 1:
                chrom[b2][tx] = 0
                chrom[b1][tx] = 1

        fixed = repair_assignment(chrom)
        return fixed if fixed is not None else chromosome

    fixed = repair_assignment(chrom)
    return fixed if fixed is not None else chromosome

# -------------------------
# GA parameters
# -------------------------
population_size = 200
generations = 150
mutation_rate = 0.15
stagnation_threshold = 25

# Time control: 10 minutes hard stop (FIXED)
max_time = 10 * 60
start_time = time.time()

# -------------------------
# Initialize population (must be feasible)
# -------------------------
population = []
while len(population) < population_size:
    c = create_chromosome()
    if c is not None:
        population.append(c)

best_solution, best_fitness = None, float("inf")
generations_since_improvement = 0

# -------------------------
# Run GA
# -------------------------
for generation in range(generations):
    if time.time() - start_time > max_time:
        print("Time limit reached (10 minutes). Stopping optimization.")
        break

    # --------- Evaluate CURRENT population ---------
    fitness_scores = [calculate_fitness(ind) for ind in population]

    if all(np.isinf(f) for f in fitness_scores):
        print(f"All individuals infeasible in generation {generation}. Re-initializing population...")
        population = []
        while len(population) < population_size:
            c = create_chromosome()
            if c is not None:
                population.append(c)
        continue

    min_fitness = min(fitness_scores)
    if min_fitness < best_fitness:
        best_fitness = min_fitness
        best_solution = [row[:] for row in population[fitness_scores.index(min_fitness)]]
        generations_since_improvement = 0
        print(f"✓ Generation {generation}: New best fitness = {best_fitness:.6f}")
    else:
        generations_since_improvement += 1

    if generations_since_improvement >= stagnation_threshold:
        print("No improvement, stopping optimization.")
        break

    # --------- Selection (roulette on inverted fitness) ---------
    safe_scores = [f if np.isfinite(f) and f > 0 else 1e12 for f in fitness_scores]
    inverted = [1.0 / f for f in safe_scores]
    total_inv = sum(inverted)
    if total_inv == 0 or not np.isfinite(total_inv):
        selection_probs = None  # fallback to uniform
    else:
        selection_probs = [x / total_inv for x in inverted]

    selected_indices = np.random.choice(
        population_size,
        size=(population_size // 2) * 2,
        p=selection_probs,
        replace=True,
    )

    # --------- Crossover (robust) ---------
    new_population = []
    for i in range(0, len(selected_indices), 2):
        parent1 = population[selected_indices[i]]
        parent2 = population[selected_indices[i + 1]]

        # Minimal robustness: ensure parents are repaired and build tx->block maps
        p1 = repair_assignment([row[:] for row in parent1])
        p2 = repair_assignment([row[:] for row in parent2])
        if p1 is None or p2 is None:
            continue

        map1 = tx_to_block_map(p1)
        map2 = tx_to_block_map(p2)
        if map1 is None or map2 is None:
            continue

        crossover_point = random.randint(1, n - 1)
        child1 = [row[:] for row in p1]
        child2 = [row[:] for row in p2]

        for tx in range(crossover_point, n):
            b1 = map1[tx]
            b2 = map2[tx]
            if b1 == b2:
                continue

            child1[b1][tx] = 0
            child2[b2][tx] = 0
            child1[b2][tx] = 1
            child2[b1][tx] = 1

        child1 = repair_assignment(child1)
        child2 = repair_assignment(child2)

        if child1 is not None:
            new_population.append(child1)
        if child2 is not None:
            new_population.append(child2)

        if len(new_population) >= population_size:
            break

    while len(new_population) < population_size:
        c = create_chromosome()
        if c is not None:
            new_population.append(c)

    # --------- Mutation ---------
    for idx in range(len(new_population)):
        if random.random() < mutation_rate:
            new_population[idx] = mutate(new_population[idx])

    # Also evaluate CHILDREN and update best_solution from them too
    children_fitness = [calculate_fitness(ind) for ind in new_population]
    if not all(np.isinf(f) for f in children_fitness):
        min_child_fit = min(children_fitness)
        if min_child_fit < best_fitness:
            best_fitness = min_child_fit
            best_solution = [row[:] for row in new_population[children_fitness.index(min_child_fit)]]
            generations_since_improvement = 0
            print(f"✓ Generation {generation}: New best fitness (children) = {best_fitness:.6f}")

    population = new_population

print("Fitness of best solution:", best_fitness)

# -------------------------
# Report
# -------------------------
if best_solution is not None:
    best_solution = repair_assignment(best_solution)

if best_solution is None or np.isinf(best_fitness):
    print("⚠️  No feasible solution found.")
else:
    block_sizes = [sum(block) for block in best_solution]
    non_empty = [size for size in block_sizes if size > 0]

    max_block_size = max(non_empty) if non_empty else 0
    print(f"\nRemark (optimal block size) = {max_block_size} transactions")
    print(f"Constraint (3) check: total assigned = {sum(block_sizes)}/{n}")

✓ Generation 0: New best fitness = 48.370559
Time limit reached (10 minutes). Stopping optimization.
Fitness of best solution: 48.370558993961744

Remark (optimal block size) = 8 transactions
Constraint (3) check: total assigned = 1000/1000


In [41]:
H_CACHE = {}